# Глубокое обучение с подкреплением

**AI4 · Other Topics** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~15 min

## Введение

Обучение с подкреплением (RL) считается одним из основных парадигм машинного обучения наряду с обучением с учителем и без учителя. Если в обучении с учителем мы опираемся на набор данных с известными результатами, то RL основано на **обучении через действие**. Например, когда мы впервые видим компьютерную игру, мы начинаем играть, даже не зная правил, и вскоре можем улучшить свои навыки просто благодаря процессу игры и корректировке своего поведения.


Для выполнения RL нам нужно:

* **Среда** или **симулятор**, который задает правила игры. Мы должны иметь возможность проводить эксперименты в симуляторе и наблюдать результаты.
* Некоторая **функция вознаграждения**, которая указывает, насколько успешным был наш эксперимент. В случае обучения игре в компьютерную игру вознаграждением будет наш итоговый счет.

На основе функции вознаграждения мы должны быть способны корректировать свое поведение и улучшать свои навыки, чтобы в следующий раз играть лучше. Основное отличие RL от других типов машинного обучения заключается в том, что в RL мы обычно не знаем, выиграли мы или проиграли, до тех пор, пока не закончим игру. Таким образом, мы не можем сказать, является ли определенный ход хорошим или плохим — вознаграждение мы получаем только в конце игры.

Во время RL мы обычно проводим множество экспериментов. В каждом эксперименте нам нужно находить баланс между следованием оптимальной стратегии, которую мы изучили до сих пор (**эксплуатация**), и исследованием новых возможных состояний (**исследование**).



## OpenAI Gym

Отличным инструментом для RL является [OpenAI Gym](https://gymnasium.farama.org/) — **среда симуляции**, которая может моделировать множество различных сред, начиная от игр Atari и заканчивая физикой балансировки шеста. Это одна из самых популярных сред симуляции для обучения алгоритмов обучения с подкреплением, поддерживаемая [OpenAI](https://openai.com/).

> **Note**: Вы можете увидеть все доступные среды OpenAI Gym [здесь](https://gymnasium.farama.org/environments/classic_control/).



## Балансировка CartPole

Вы, вероятно, видели современные устройства для балансировки, такие как *Segway* или *гироскутеры*. Они способны автоматически балансировать, регулируя свои колеса в ответ на сигнал от акселерометра или гироскопа. В этом разделе мы научимся решать похожую задачу — балансировку шеста. Это похоже на ситуацию, когда цирковой артист балансирует шест на своей руке, но в данном случае балансировка происходит только в одной плоскости.

Упрощенная версия задачи балансировки известна как проблема **CartPole**. В мире CartPole у нас есть горизонтальная платформа, которая может двигаться влево или вправо, и цель — сбалансировать вертикальный шест на вершине платформы, пока она движется.

![a cartpole](ms-assets/ai/cartpole.f52a67f27e058170.webp)

Чтобы создать и использовать эту среду, нам нужно всего несколько строк кода на Python:



In [ ]:
import gym
env = gym.make("CartPole-v1")

env.reset()
done = False
total_reward = 0
while not done:
   env.render()
   action = env.action_space.sample()
   observaton, reward, done, info = env.step(action)
   total_reward += reward

print(f"Total reward: {total_reward}")

Каждая среда может быть доступна точно таким же образом:
* `env.reset` начинает новый эксперимент
* `env.step` выполняет шаг симуляции. Он принимает **действие** из **пространства действий** и возвращает **наблюдение** (из пространства наблюдений), а также вознаграждение и флаг завершения.

В приведенном выше примере на каждом шаге выполняется случайное действие, поэтому продолжительность эксперимента очень короткая:

![non-balancing cartpole](ms-assets/ai/lessons/6-Other/22-DeepRL/images/cartpole-nobalance.gif)

Цель алгоритма RL — обучить модель, так называемую **политику** π, которая будет возвращать действие в ответ на данное состояние. Мы также можем рассматривать политику как вероятностную, например, для любого состояния *s* и действия *a* она будет возвращать вероятность π(*a*|*s*), что мы должны выполнить *a* в состоянии *s*.



## Алгоритм Policy Gradients

Самый очевидный способ моделирования политики — создание нейронной сети, которая будет принимать состояния в качестве входных данных и возвращать соответствующие действия (или скорее вероятности всех действий). В некотором смысле это похоже на обычную задачу классификации, с одним важным отличием — мы заранее не знаем, какие действия следует предпринимать на каждом из шагов.

Идея заключается в оценке этих вероятностей. Мы строим вектор **накопленных вознаграждений**, который показывает наш общий результат на каждом шаге эксперимента. Мы также применяем **дисконтирование вознаграждений**, умножая вознаграждение, полученное через t шагов, на γ^t (γ=0.99): чем позже награда, тем меньше её вес. Затем мы усиливаем те шаги на пути эксперимента, которые приносят больше вознаграждений.

> Узнайте больше об алгоритме Policy Gradient и посмотрите его в действии в [примере ноутбука](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL/CartPole-RL-TF.ipynb).



## Алгоритм Actor-Critic

Улучшенная версия подхода Policy Gradients называется **Actor-Critic**. Основная идея заключается в том, что нейронная сеть будет обучена возвращать две вещи:

* Политику, которая определяет, какое действие предпринять. Эта часть называется **actor**.
* Оценку общего вознаграждения, которое мы можем ожидать в данном состоянии — эта часть называется **critic**.

В некотором смысле эта архитектура напоминает [GAN](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/4-ComputerVision/10-GANs/README.md), где у нас тоже две сети, но они не соревнуются, а дополняют друг друга. В модели Actor-Critic актор предлагает действие, которое нужно предпринять, а критик пытается быть критичным и оценить результат. Однако наша цель — обучить эти сети в унисон.

Поскольку мы знаем как реальные накопленные вознаграждения, так и результаты, возвращаемые критиком во время эксперимента, относительно легко построить функцию потерь, которая минимизирует разницу между ними. Это даст нам **потери критика**. Мы можем вычислить **потери актора**, используя тот же подход, что и в алгоритме Policy Gradient.

После выполнения одного из этих алгоритмов мы можем ожидать, что наш CartPole будет вести себя следующим образом:

![a balancing cartpole](ms-assets/ai/lessons/6-Other/22-DeepRL/images/cartpole-balance.gif)



## Упражнения: Policy Gradients и Actor-Critic RL

Продолжите обучение в следующих ноутбуках:

* [RL в TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL/CartPole-RL-TF.ipynb)
* [RL в PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL/CartPole-RL-PyTorch.ipynb)



## Другие задачи RL

Обучение с подкреплением сегодня является быстро развивающейся областью исследований. Некоторые интересные примеры обучения с подкреплением:

* Обучение компьютера играть в **игры Atari**. Сложность этой задачи заключается в том, что у нас нет простого состояния, представленного в виде вектора, а есть скриншот — и нам нужно использовать CNN, чтобы преобразовать изображение экрана в вектор признаков или извлечь информацию о вознаграждении. Игры Atari доступны в Gym.
* Обучение компьютера играть в настольные игры, такие как шахматы и го. Недавно программы, такие как **Alpha Zero**, были обучены с нуля двумя агентами, играющими друг против друга и улучшающимися на каждом шаге.
* В промышленности RL используется для создания систем управления на основе симуляции. Сервис [Bonsai](https://azure.microsoft.com/services/project-bonsai/?WT.mc_id=academic-77998-cacaste) специально разработан для этого.



## Заключение

Мы научились обучать агентов достигать хороших результатов, просто предоставляя им функцию вознаграждения, которая определяет желаемое состояние игры, и давая им возможность разумно исследовать пространство поиска. Мы успешно попробовали два алгоритма и достигли хорошего результата за относительно короткий период времени. Однако это лишь начало вашего пути в RL, и вам определенно стоит рассмотреть возможность прохождения отдельного курса, если вы хотите углубиться в эту тему.



## Задание

Изучите приложения, перечисленные в разделе «Другие задачи RL», и попробуйте реализовать одно из них!



## Обзор и самостоятельное изучение

Узнайте больше о классическом обучении с подкреплением в нашем [курсе «Машинное обучение для начинающих»](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/README.md).

Посмотрите [это отличное видео](https://www.youtube.com/watch?v=qv6UVOQ0F44) о том, как компьютер может научиться играть в Super Mario.



## Задание: Обучите Mountain Car

Ваша цель в этом задании — обучить другую среду Gym — [Mountain Car](https://gymnasium.farama.org/).



---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/6-Other/22-DeepRL)
- [Учебник ML (ШАД): обучение с подкреплением](https://contest.yandex.ru/tracks/ml/environment-interaction/reinforcement-learning)